<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Costos logísticos internos y externos de transporte

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `funciones.fuzzy_match`: emparejamiento difuso de nombres de departamentos y municipios.
- `funciones.loc_ext_iso_groups`: mapeo de nombres de país a grupos regionales y códigos ISO3.

In [1]:
import pandas as pd
import numpy as np
from funciones.fuzzy_match import fuzzy_match
from funciones.loc_ext_iso_groups import name_to_group, name_to_iso3

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

Define los percentiles que delimitan los valores atípicos del costo por kilogramo.

In [3]:
percentil_inferior = 0.005
percentil_superior = 0.995

### Lectura de datos

Carga la base del Registro Nacional de Despachos de Carga, las declaraciones agregadas del
formulario 600 y las tablas de correspondencia arancelaria a cuatro dígitos.

In [4]:
df_rndc = pd.read_parquet(raw / 'logistica' / 'rndc.parquet')
df_600 = pd.read_parquet(temp / 'df_600.parquet')

df_hs2012_2017 = pd.read_parquet(temp / 'df_hs2012_2017.parquet')
df_hs4_12_17 = df_hs2012_2017.copy()
df_hs4_12_17['HS4'] = df_hs4_12_17['HS6'].str[:4]
df_hs4_12_17['HS4_17'] = df_hs4_12_17['HS6_17'].str[:4]
df_hs4_12_17 = df_hs4_12_17.drop(columns=['HS6_17','HS6']).drop_duplicates()

df_hs2012_2022 = pd.read_parquet(temp / 'df_hs2012_2022.parquet')
df_hs2012_2022_4 = df_hs2012_2022.copy()
df_hs2012_2022_4['HS4_22'] = df_hs2012_2022_4['HS6_22'].str[:4]
df_hs2012_2022_4['HS4'] = df_hs2012_2022_4['HS6'].str[:4]
df_hs2012_2022_4 = df_hs2012_2022_4.drop(columns=['HS6_22','HS6']).drop_duplicates()

### Costo logístico interno por kilogramo

Estima el costo del transporte terrestre de carga dentro del país.

**Pasos realizados:**

1. **Conversión de galones a kilogramos:** calcula un factor por tipo de mercancía con los registros que reportan ambas magnitudes, e imputa el peso de los que solo reportan galones.

2. **Limpieza de municipios:** retira del nombre del municipio de destino la porción que repite el departamento.

3. **Estandarización del código de mercancía:** lo lleva a cuatro dígitos y lo cruza con la correspondencia HS 2022 a HS 2012.

4. **Emparejamiento con el formulario 600:** alinea departamentos de origen y municipios de destino con las categorías del formulario mediante emparejamiento difuso, y descarta los registros sin correspondencia.

5. **Costo por kilogramo:** divide el valor pagado entre el peso transportado y descarta los registros inconsistentes.

6. **Recorte de atípicos:** elimina los valores fuera del rango intercuantílico definido en los parámetros, por combinación de mercancía, origen y destino.

In [5]:
df_galones_a_kg = df_rndc.query("GALONES > 0 and KILOGRAMOS > 0").groupby(['MERCANCIA'])[['GALONES', 'KILOGRAMOS']].sum().reset_index()
df_galones_a_kg['GALONES_A_KILOGRAMOS'] = df_galones_a_kg['KILOGRAMOS'] / df_galones_a_kg['GALONES']
df_galones_a_kg = df_galones_a_kg.groupby(['MERCANCIA'])[['GALONES_A_KILOGRAMOS']].mean().reset_index()

df_rndc = df_rndc.merge(df_galones_a_kg[['MERCANCIA', 'GALONES_A_KILOGRAMOS']], on='MERCANCIA', how='left')
df_rndc['GALONES_A_KILOGRAMOS'] = np.where(
    (df_rndc['GALONES'] > 0) & (df_rndc['KILOGRAMOS'] == 0),
    df_rndc['GALONES_A_KILOGRAMOS'],
    np.nan
)
df_rndc['KILOGRAMOS'] = np.where(
    (df_rndc['GALONES'] > 0) & (df_rndc['KILOGRAMOS'] == 0),
    df_rndc['GALONES'] * df_rndc['GALONES_A_KILOGRAMOS'],
    df_rndc['KILOGRAMOS']
).round()

df_rndc['MUNICIPIODESTINO'] = df_rndc.apply(
    lambda row: str(row['MUNICIPIODESTINO']).replace(str(row['DEPARTAMENTODESTINO']), '').strip()
                if pd.notnull(row['MUNICIPIODESTINO']) and pd.notnull(row['DEPARTAMENTODESTINO'])
                else row['MUNICIPIODESTINO'],
    axis=1
)

df_rndc['CODMERCANCIA'] = pd.to_numeric(df_rndc['CODMERCANCIA'].str.strip(), errors='coerce').astype('Int64').astype('str').str.zfill(4)

df_rndc = df_rndc.merge(df_hs2012_2022_4.rename(columns={'HS4':'CODMERCANCIA'}), how='left', left_on='CODMERCANCIA', right_on='HS4_22', suffixes=('_22', '')).dropna(subset=['CODMERCANCIA']).reset_index(drop=True)

df_rndc['CODMERCANCIA2'] = df_rndc['CODMERCANCIA'].astype(str).str[:2].str.zfill(2)

choices = df_600['REGION_PROCEDENCIA'].dropna().unique()
choices_aduana = df_600['ADUANA_DESPACHO'].dropna().unique()
df_rndc['DEPARTAMENTOORIGEN_match'] = df_rndc['DEPARTAMENTOORIGEN'].apply(lambda x: fuzzy_match(x.title(), choices) if isinstance(x, str) else None)
df_rndc['MUNICIPIODESTINO_match'] = df_rndc['MUNICIPIODESTINO'].apply(lambda x: fuzzy_match(x.title(), choices_aduana) if isinstance(x, str) else None)
df_rndc.dropna(subset=['MUNICIPIODESTINO_match'], inplace=True)
df_rndc.dropna(subset=['DEPARTAMENTOORIGEN_match'], inplace=True)

df_rndc['COSTO POR KG'] = df_rndc['VALORESPAGADOS'] / df_rndc['KILOGRAMOS']
df_rndc = df_rndc.query("VALORESPAGADOS > 0 and KILOGRAMOS > 0 and VIAJESTOTALES != KILOGRAMOS and CODMERCANCIA != '<NA>'")

df_rndc['lower_bound'] = df_rndc.groupby(['CODMERCANCIA2', 'DEPARTAMENTOORIGEN_match', 'MUNICIPIODESTINO_match'])['COSTO POR KG']\
                                .transform(lambda x: x.quantile(percentil_inferior))
df_rndc['upper_bound'] = df_rndc.groupby(['CODMERCANCIA2', 'DEPARTAMENTOORIGEN_match', 'MUNICIPIODESTINO_match'])['COSTO POR KG']\
                                .transform(lambda x: x.quantile(percentil_superior))

df_rndc_filtered = df_rndc[(df_rndc['COSTO POR KG'] >= df_rndc['lower_bound']) &
                           (df_rndc['COSTO POR KG'] <= df_rndc['upper_bound'])].copy()

temp.mkdir(parents=True, exist_ok=True)
df_rndc_filtered.to_parquet(temp / 'df_rndc.parquet', index=False)

### Costo externo de transporte internacional

Consolida el costo promedio por kilogramo transportado entre países, por modo de transporte y
producto a cuatro dígitos.

El insumo es la base de costos de transporte de Estados Unidos, que no se reconstruye en esta
cadena: se archiva en `data/raw/logistica/` con su fecha de descarga.

**Pasos realizados:**

1. **Estandarización de países:** asigna el grupo geoeconómico del importador y los códigos ISO3 de importador y exportador.

2. **Homologación arancelaria:** traduce el código HS4 de 2017 a la nomenclatura de 2012.

3. **Agregación:** suma el gasto en transporte y el peso neto por año, modo, origen, destino y producto.

4. **Costo por kilogramo:** divide el gasto entre el peso y descarta los resultados infinitos o nulos.

In [6]:
# el original leia este archivo desde ~/Documents, ruta que ya no existe.
# la copia archivada vive en data/raw/logistica/
df_costos_externos = pd.read_parquet(raw / 'logistica' / 'df_costos_externos.parquet').dropna().reset_index(drop=True)

df_costos_externos["Grupo Importador"] = df_costos_externos["Importador"].map(name_to_group).fillna("Otro")
df_costos_externos["ISO Importador"]  = df_costos_externos["Importador"].map(name_to_iso3).fillna("UNK")
df_costos_externos["ISO Exportador"]  = df_costos_externos["Exportador"].map(name_to_iso3).fillna("UNK")

df_costos_externos = df_costos_externos.merge(df_hs4_12_17.rename(columns={'HS4':'HS4_12'}), how='left', left_on='HS4', right_on='HS4_17').drop(columns=['HS4_17','HS4']).rename(columns={'HS4_12':'HS4'}).reset_index(drop=True)

df_costos_externos = df_costos_externos.groupby(['Año', 'Transporte', 'Exportador','ISO Exportador', 'Importador', 'ISO Importador', 'Grupo Importador', 'HS4'])[['Gasto', 'KG']].sum().reset_index()

df_costos_externos['Valor por kg'] = df_costos_externos['Gasto'] / df_costos_externos['KG']

df_costos_externos = df_costos_externos[['HS4', 'Transporte', 'Importador', 'ISO Importador', 'Exportador','ISO Exportador', 'Grupo Importador', 'Valor por kg']].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

temp.mkdir(parents=True, exist_ok=True)
df_costos_externos.to_parquet(temp / 'df_costos_externos.parquet', index=False)